In [68]:
HF_HUB_OFFLINE = 1
from huggingface_hub import scan_cache_dir
from datasets import load_dataset, concatenate_datasets, Dataset
from pprint import pprint

### Load the pinned revision

In [ ]:
cache = scan_cache_dir()

for repo in cache.repos:
    if repo.repo_type=="dataset" and repo.repo_id=="HuggingFaceBio/FineDNA":
        for revision in repo.revisions:
            print(f"Commit: {revision.commit_hash}")
            print(f"Refs: {revision.refs}")
            REVISION = revision.commit_hash

Commit: e918c3649ffc614684d10e46e1eb5ba41170e7f9
Refs: frozenset({'main'})


In [30]:
REVISION = "e918c3649ffc614684d10e46e1eb5ba41170e7f9"

#load the configs from our fixed revision into local cache
configs = ["sft_sequence_classification",
"sft_dna_computation",
"rl_mix",
]

original_datasets = {name : load_dataset("HuggingFaceBio/FineDNA", name, split="train",revision=REVISION ,streaming=False) for name in configs}

### Add Stable-IDs before filtering & shuffling

In [75]:
# Add stableIDs to our dataset before creating the train-val-test splits
def add_identifiers(example, source_index, revision, config):
    """ source_index = the row number inside the original loaded dataset"""
    
    return {
        'revision': revision,
        'source_index': source_index,
        'example_id': f"{config}/train/{source_index}"
    }
datasets = {}
for config in original_datasets:
    datasets[config] = original_datasets[config].map(add_identifiers, with_indices=True,
                                             fn_kwargs={"revision": REVISION, "config": config})

for row in datasets["rl_mix"].select(range(3)):
    pprint(row)

{'answer_text': 'benign',
 'domain': 'variant_effect_prediction',
 'example_id': 'rl_mix/train/0',
 'gold_answer_json': None,
 'prompt': [{'content': 'Determine whether this human single-nucleotide '
                        'variant is benign or pathogenic from the DNA '
                        'sequence(s) provided.\n'
                        '\n'
                        'The variant is at position 4680 in the reference '
                        'sequence, using 1-based indexing. At this position, '
                        'the reference allele A is changed to the alternate '
                        'allele G.\n'
                        '\n'
                        'Reference DNA sequence:\n'
                        '\n'
                        'AAATCAATATCACAAAAAGAATACCCCATTCTTATGCTTAGTTTTCCATTATACAGATAGTGCCTTATTATTGGTGGACACTTAGATTATTTCTAGGTTTTAGTCACCAGACACAACGCTGCAGTGAACACTGTATTTTTGCCCAAATATGTGAATATAATGGTAGGGTAAAGAGAGAACACAACTCACTCTAGCTTTGGTTTCTTGCAGGCTTTCCTTGAGTTGGTCTCTCTCCAGTTTGAG

### Check that the IDs have been added to all rows and that they map correctly to the loaded revisions of FineDNA

In [84]:
from typing import Any
import numpy as np
# check for missing rows, uniqueness of ids, etc.

def _check_missing_id(dataset):
    missing_count = 0
    for row in dataset:
        id = row.get("example_id")
        if not id or id == "":
            missing_count += 1
    return missing_count

def _check_unique_ids(dataset):
    ids = dataset["example_id"]
    return len(ids)==len(set(ids))

def _check_ids_resolve_correctly(config_name, config_ds, source_datasets):
    rng = np.random.default_rng(seed=40)
    random_indices = rng.integers(0, len(config_ds), size=10)
    resolve_bool = True

    for row_num in random_indices:
        new_row = config_ds[row_num]
        source_index = new_row.get("source_index")
        original_row = source_datasets[config_name][source_index]
        fields_to_check = {key: new_row[key] for key in original_row.keys()}

        if fields_to_check != original_row:
            print(f"Row {row_num} failed resolving")
            resolve_bool = False
        elif new_row.get("example_id") != f"{config_name}/train/{source_index}":
            print(f"Row {row_num} has wrong example_id")
            resolve_bool = False

    return resolve_bool


def _check_audit_ids_resolve_correctly(config_name, config_ds, source_datasets):
    rng = np.random.default_rng(seed=40)
    random_indices = rng.integers(0, len(config_ds), size=10)
    resolve_bool = True

    for row_num in random_indices:
        new_row = config_ds[row_num]
        source_index = new_row.get("source_index")
        source = source_datasets[config_name]
        original_row = source[source_index]

        source_id = f"{config_name}/train/{source_index}"
        parent_id = new_row.get("parent_id")

        fields_to_check = {
            "example_id": parent_id if parent_id else new_row["example_id"],
            "domain": new_row["domain"],
        }
        expected_fields = {
            "example_id": source_id,
            "domain": original_row["domain"],
        }

        if config_name.startswith("rl_"):
            fields_to_check["task_group"] = new_row.get("task_group")
            fields_to_check["task_type"] = new_row.get("task_type")
            fields_to_check["prompt"] = new_row.get("model_input")
            expected_fields.update({
                "task_group": original_row.get("task_group"),
                "task_type": original_row.get("task_type"),
                "prompt": original_row["prompt"],
            })
            if parent_id is not None or new_row["example_id"] != source_id:
                print(f"Row {row_num} has wrong RL example_id or parent_id")
                resolve_bool = False

        elif config_name.startswith("sft_"):
            child_prefix = source_id + "/assistant/"
            child_id = new_row["example_id"]
            turn = child_id[len(child_prefix):] if child_id.startswith(child_prefix) else ""
            assistant_positions = [i for i, message in enumerate(original_row["messages"]) if message["role"] == "assistant"]
            if parent_id != source_id or not turn.isdigit() or not 1 <= int(turn) <= len(assistant_positions):
                print(f"Row {row_num} has wrong SFT example_id or parent_id")
                resolve_bool = False
                continue
            fields_to_check["messages"] = new_row.get("model_input")
            expected_fields["messages"] = original_row["messages"][:assistant_positions[int(turn) - 1]]

        if fields_to_check != expected_fields:
            print(f"Row {row_num} failed resolving")
            resolve_bool = False

    return resolve_bool

In [85]:
#combine above helpers into a single verification function
def verify_stable_ids(datasets, source_datasets: dict[Any, Any], is_audit:bool=False):
    missing_count = 0
    unique_ids = True
    resolve_bool = False

    for name, config in datasets.items():
        missing_count += _check_missing_id(config)
        
        unique_ids = _check_unique_ids(config)
        if unique_ids==False:
            print(f"Config {name} failed uniqueness check!\n")

        if is_audit:
            resolve_bool = _check_audit_ids_resolve_correctly(name, config, source_datasets)
        else:
            resolve_bool = _check_ids_resolve_correctly(name, config, source_datasets)
        if resolve_bool == False:
            print(f"Config {name} failed 'resolving to source' check!\n")

    #check for uniqueness of ids across all configs
    ds_concat = concatenate_datasets([dataset for dataset in datasets.values()])
    unique_ids = _check_unique_ids(ds_concat)

    print(f"Missing Count: {missing_count}")
    print(f"Unique Ids: {unique_ids}")
    print(f"IDs resolve to source: {resolve_bool}")

In [86]:
verify_stable_ids(datasets, original_datasets)

Missing Count: 0
Unique Ids: True
IDs resolve to source: True


In [50]:
verifiable_domains = ["dna_computation", "sequence_classification", 
                  "biology_qa"]

datasets["rl_mix"] = datasets["rl_mix"].filter(lambda row: row['domain'] in verifiable_domains)

#verify that the rl_mix retains stable ids post filtering also
verify_stable_ids(datasets, original_datasets)

Filter:   0%|          | 0/98433 [00:00<?, ? examples/s]

Filter: 100%|██████████| 98433/98433 [00:00<00:00, 105697.47 examples/s]


Missing Count: 0
Unique Ids: True
IDs resolve to source: True


### Save basic dataset properties in CSV

In [51]:
import pandas as pd 
from pathlib import Path

def save_datasets_csv(datasets: dict[Any, Any], settings_dir):
    records =[]
    for name, dataset in datasets.items():
        info = {
            "source": name,
            "revision": REVISION,
            "original_rows": len(original_datasets[name]),
            "after_domain_filtering_row": len(dataset),
            "domain_filters": verifiable_domains if name == "rl_mix" else None
        }
        records.append(info)
    df = pd.DataFrame(records)
    df.to_csv(Path(settings_dir/"datasets-snapshot.csv"))

settings_dir = Path("./settings")
settings_dir.mkdir(exist_ok=True, parents=True)
save_datasets_csv(datasets, settings_dir)

### Build an audit table to detect overlaps & clean data before creating `test-val-train` splits

In [60]:
# cleaning up formatting for each conversation history
def normalize_messages(model_input):
    normalized = []

    for message in model_input:
        content = message["content"]

        # Standardize line endings, then remove outer whitespace.
        content = content.replace("\r\n", "\n")
        content = content.replace("\r", "\n")
        content = content.strip()

        normalized.append({
            "role": message["role"],
            "content": content,
        })

    return normalized

# helper to build conversation history until desired assistant turn number
def _build_message_history(messages: list[dict], assistant_turns: int):
    counter = 0 
    msg_history = []
    for message in messages:
        #first increment the assistant turn
        if message["role"] == "assistant":
            counter +=1
        #if we've reached the point to splice at, break.
        if counter == assistant_turns:
            break
        msg_history.append(message)
    return msg_history

In [88]:
"""
Audit Table Fields:
| Field | SFT | RL |
| `example_id`, `source_config`, `source_index`, `domain` | From the indexed dataset | From the indexed dataset |
| `task_group`, `task_type` | Use `unknown` where unavailable | From the source fields; use `unknown` for missing values |
| `model_input` | Conversation prefix before the target assistant answer | `prompt` |
"""     

# break down multi-turn conversations into childrens with new example_ids
def _audit_extract_sft_rows(config_name, ds):
    records = []
    for row in ds:
        user_turns = 0
        assistant_turns = 0

        #iterate over the multi-turn message lists and break them into children entries
        for message in row["messages"]:
            if message["role"] == "user":
                user_turns += 1

            if message["role"] == "assistant":
                assistant_turns += 1
                model_input = _build_message_history(row["messages"], assistant_turns)
                child = {
                    "example_id": row["example_id"] + f"/assistant/{assistant_turns}",
                    "parent_id": row["example_id"],
                    "source_config": config_name,
                    "domain": row["domain"],
                    "source_index": row["source_index"],
                    "task_group": row.get("task_group", None),
                    "task_type": row.get("task_type", None),
                    "model_input": model_input,
                    "normalized_input": normalize_messages(model_input)
                }
                records.append(child)
    return Dataset.from_list(records)

def _audit_extract_rl_rows(config_name, ds):
    records = []
    for row in ds:
        entry = {
                "example_id": row["example_id"],
                "parent_id": None,
                "source_config": config_name,
                "domain": row["domain"],
                "source_index": row["source_index"],
                "task_group": row.get("task_group", None),
                "task_type": row.get("task_type", None),
                "model_input": row["prompt"],
                "normalized_input": normalize_messages(row["prompt"])
            }
        records.append(entry)
    return Dataset.from_list(records)


def build_audit_table(datasets):
    audit_ds = Dataset.from_list([])
    for (name, config) in datasets.items():
        
        if name.startswith("sft_"):
            new_additions =  _audit_extract_sft_rows(name, config)
        
        elif name.startswith("rl_"):
            new_additions = _audit_extract_rl_rows(name, config)

        audit_ds = concatenate_datasets([audit_ds, new_additions])
        verify_stable_ids({name: new_additions}, {name:config}, True)


    return audit_table

audit_table = build_audit_table(datasets)

Missing Count: 0
Unique Ids: True
IDs resolve to source: True
Missing Count: 0
Unique Ids: True
IDs resolve to source: True
Missing Count: 0
Unique Ids: True
IDs resolve to source: True
